# 01 – PSMA PET/CT Segmentation Demo

**SNMMI AI Certificate – Hands-On Module**

In this notebook you will:
- Download a PSMA PET/CT 2-D patch dataset
- Train a compact 2-D U-Net for 2–3 epochs
- Evaluate Dice/IoU on a validation split
- Compare the DL model against a threshold-based baseline (40 % of per-patch max SUV)

> **Run cells top-to-bottom on a fresh Colab runtime. Do not skip cells.**

---

**Dataset citation:**

> Gatidis, S., Kuestner, T., Ingrisch, M., Hepp, T., Frueh, M., Nikolaou, K.,
> La Fougere, C., Pfannenberg, C., Fabritius, M., Jeblick, K., Schachtner, B.,
> Dexl, J., Wesp, P., Mittermeier, A., Unterrainer, L., Sheikh, G., Boening, G.,
> Brendel, M., Ricke, J., … Cyran, C. (2025).
> *PSMA-FDG-PET-CT-Lesions*.
> University of Tuebingen, Ludwig-Maximilians-University Munich.
> [https://doi.org/10.57754/FDAT.0zs4c-89f12](https://doi.org/10.57754/FDAT.0zs4c-89f12)
>
> License: **CC BY-NC 4.0** – non-commercial use only.

## 0 · Setup & Config

In [ ]:
%pip install -q numpy matplotlib tqdm scikit-learn
# torch and torchvision are pre-installed on Colab; kept for non-Colab users

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS     = 3       # keep ≤ 3 for free Colab
BATCH_SIZE = 16
LR         = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1 · Download & Unzip PSMA Sample Data

In [ ]:
# ── Paste your GitHub Release URL for the PSMA zip here ──────────────────────
DATA_URL = "https://github.com/ZekunLi-Zeke/snmmi-ai-hands-on-segmentation/releases/download/v0.1/psma_sample.zip"
# ─────────────────────────────────────────────────────────────────────────────

import pathlib, urllib.request, zipfile

DATA_DIR = pathlib.Path("./data/psma")
DATA_DIR.mkdir(parents=True, exist_ok=True)

zip_path = pathlib.Path("/tmp/psma_sample.zip")

if not any(DATA_DIR.rglob("*.npz")):
    if not zip_path.exists():
        print("Downloading PSMA sample data …")
        urllib.request.urlretrieve(DATA_URL, zip_path)
        print("Download complete.")
    print("Unzipping …")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already present.")

npz_files = sorted(DATA_DIR.rglob("*.npz"))
assert len(npz_files) > 0, "No .npz file found – check DATA_URL and re-run."
print(f"Found: {[f.name for f in npz_files]}")

# Quick peek at array shapes
data = np.load(npz_files[0])
print(f"Keys: {list(data.keys())}")
for k in data:
    print(f"  {k}: shape={data[k].shape}  dtype={data[k].dtype}")

## 2 · Load Data & Visualise Patches

In [ ]:
class PatchDataset(Dataset):
    """2-D PET/CT patch dataset from a .npz file.

    Expected .npz keys
    ------------------
    pet  : (N, 64, 64) float32 – SUV values (already normalised 0-1 or raw SUV)
    ct   : (N, 64, 64) float32 – HU values (normalised to 0-1)
    mask : (N, 64, 64) float32 – binary lesion mask {0, 1}
    """

    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path)
        self.pet    = data["pet"].astype(np.float32)
        self.ct     = data["ct"].astype(np.float32)
        self.mask   = data["mask"].astype(np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.pet)

    def __getitem__(self, idx):
        pet  = self.pet[idx].copy()
        ct   = self.ct[idx].copy()
        mask = self.mask[idx].copy()

        if self.augment and random.random() > 0.5:
            # Random horizontal flip
            pet  = pet[:, ::-1].copy()
            ct   = ct[:, ::-1].copy()
            mask = mask[:, ::-1].copy()

        x = np.stack([pet, ct], axis=0)  # (2, 64, 64) – 2-channel input
        y = mask[np.newaxis]              # (1, 64, 64) – binary target
        return torch.from_numpy(x), torch.from_numpy(y)


npz_path = sorted(DATA_DIR.rglob("*.npz"))[0]
full_ds  = PatchDataset(npz_path)
n_total  = len(full_ds)
n_val    = max(1, int(0.2 * n_total))
n_train  = n_total - n_val

train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Dataset: {n_total} patches  |  train {n_train}  val {n_val}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Visualise a few patches
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for col in range(4):
    x, y = full_ds[col]
    axes[0, col].imshow(x[0].numpy(), cmap="hot");   axes[0, col].set_title(f"PET #{col}")
    axes[1, col].imshow(x[1].numpy(), cmap="gray");  axes[1, col].set_title(f"CT #{col}")
    axes[2, col].imshow(y[0].numpy(), cmap="gray");  axes[2, col].set_title(f"Mask #{col}")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Sample patches (PET / CT / Mask)", fontsize=13)
plt.tight_layout()
plt.show()

## 3 · Define Compact 2-D U-Net

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3×3 conv → BN → ReLU blocks."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet2D(nn.Module):
    """Compact 2-D U-Net with skip connections (4 resolution levels)."""

    def __init__(self, in_ch=2, out_ch=1, features=16):
        super().__init__()
        f = features
        # Encoder
        self.enc1 = DoubleConv(in_ch, f)
        self.enc2 = DoubleConv(f,     f * 2)
        self.enc3 = DoubleConv(f * 2, f * 4)
        self.pool  = nn.MaxPool2d(2)
        # Bottleneck
        self.bottleneck = DoubleConv(f * 4, f * 8)
        # Decoder
        self.up3  = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(f * 8, f * 4)
        self.up2  = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(f * 4, f * 2)
        self.up1  = nn.ConvTranspose2d(f * 2, f,     kernel_size=2, stride=2)
        self.dec1 = DoubleConv(f * 2, f)
        # Output head (1×1 conv → raw logits)
        self.head = nn.Conv2d(f, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)   # shape: (B, 1, H, W), raw logits


model   = UNet2D(in_ch=2, out_ch=1, features=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters: {n_params:,}")

## 4 · Define Loss, Optimizer, & Optional Augmentation

In [ ]:
def dice_loss(logits, target, smooth=1.0):
    """Differentiable soft Dice loss (applied after sigmoid)."""
    pred = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1.0 - ((2.0 * inter + smooth) / (denom + smooth)).mean()


def combined_loss(logits, target):
    """Dice loss + Binary Cross-Entropy (gives stable gradients)."""
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    return dice_loss(logits, target) + bce


optimizer = optim.Adam(model.parameters(), lr=LR)
print("Loss: Dice + BCE  |  Optimizer: Adam  |  LR:", LR)

## 5 · Train for 2–3 Epochs & Validate

In [ ]:
def dice_score(logits, target, threshold=0.5):
    """Dice coefficient metric (non-differentiable, used for reporting)."""
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2.0 * inter + 1.0) / (denom + 1.0)).mean().item()


history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = combined_loss(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= n_train

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_dice_sum = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            val_dice_sum += dice_score(model(x), y) * x.size(0)
    val_dice = val_dice_sum / n_val

    history["train_loss"].append(epoch_loss)
    history["val_dice"].append(val_dice)
    print(f"Epoch {epoch:>2}: train loss={epoch_loss:.4f}  val Dice={val_dice:.4f}")

# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(range(1, EPOCHS + 1), history["train_loss"], marker="o")
ax1.set(title="Train Loss", xlabel="Epoch", ylabel="Loss")
ax2.plot(range(1, EPOCHS + 1), history["val_dice"], marker="o", color="orange")
ax2.set(title="Validation Dice", xlabel="Epoch", ylabel="Dice")
plt.tight_layout()
plt.show()

## 6 · Visualise Predictions vs Ground Truth

In [ ]:
model.eval()
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.5 * n_show))

with torch.no_grad():
    for row in range(n_show):
        x, y = val_ds[row]
        logit = model(x.unsqueeze(0).to(DEVICE))
        pred  = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()

        axes[row, 0].imshow(x[0].numpy(), cmap="hot");  axes[row, 0].set_title("PET")
        axes[row, 1].imshow(x[1].numpy(), cmap="gray"); axes[row, 1].set_title("CT")
        axes[row, 2].imshow(y[0].numpy(), cmap="gray"); axes[row, 2].set_title("GT Mask")
        axes[row, 3].imshow(pred,          cmap="gray"); axes[row, 3].set_title("DL Prediction")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Predictions vs Ground Truth", fontsize=14)
plt.tight_layout()
plt.show()

## 7 · Task-Based Evaluation

We evaluate two segmentation approaches:
1. **DL model** – U-Net trained above
2. **Threshold baseline** – pixels where SUV > 40 % of per-patch max SUV

For each patch we report:
- **Tumor area** – number of pixels classified as lesion (a surrogate for total tumour burden)
- **Dice coefficient** – overlap between prediction and GT mask

In [ ]:
def threshold_segment(pet_patch, fraction=0.4):
    """Threshold segmentation at *fraction* × per-patch max SUV.

    Args:
        pet_patch : 2-D numpy array (64×64) of PET/SUV values.
        fraction  : threshold fraction (default 0.40 = 40 % of max).
    Returns:
        Binary mask (64×64) numpy array {0.0, 1.0}.
    """
    max_suv = pet_patch.max()
    if max_suv == 0:
        return np.zeros_like(pet_patch)
    return (pet_patch > fraction * max_suv).astype(np.float32)


def arr_dice(a, b, smooth=1.0):
    """Dice coefficient between two numpy binary arrays."""
    inter = (a * b).sum()
    return (2.0 * inter + smooth) / (a.sum() + b.sum() + smooth)


# Evaluate on the full validation set
model.eval()
gt_areas, dl_areas, thr_areas = [], [], []
dl_dices,  thr_dices           = [], []

with torch.no_grad():
    for i in range(len(val_ds)):
        x, y      = val_ds[i]
        pet_np    = x[0].numpy()
        gt_mask   = y[0].numpy()

        logit     = model(x.unsqueeze(0).to(DEVICE))
        dl_pred   = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()
        thr_mask  = threshold_segment(pet_np)

        gt_areas.append(int(gt_mask.sum()))
        dl_areas.append(int(dl_pred.sum()))
        thr_areas.append(int(thr_mask.sum()))
        dl_dices.append(arr_dice(dl_pred, gt_mask))
        thr_dices.append(arr_dice(thr_mask, gt_mask))

# Print a sample table (first 8 patches)
print(f"{'#':>3} | {'GT area':>8} | {'DL area':>8} | {'Thr area':>9} | {'DL Dice':>8} | {'Thr Dice':>9}")
print("-" * 62)
for i in range(min(8, len(val_ds))):
    print(f"{i+1:>3} | {gt_areas[i]:>8} | {dl_areas[i]:>8} | {thr_areas[i]:>9} | "
          f"{dl_dices[i]:>8.3f} | {thr_dices[i]:>9.3f}")

print(f"\nMean DL Dice:        {np.mean(dl_dices):.3f}")
print(f"Mean Threshold Dice: {np.mean(thr_dices):.3f}")

# Visualise DL vs threshold for one sample
i = 0
x, y     = val_ds[i]
pet_np   = x[0].numpy()
gt_mask  = y[0].numpy()
thr_mask = threshold_segment(pet_np)
with torch.no_grad():
    dl_pred = (torch.sigmoid(model(x.unsqueeze(0).to(DEVICE))) > 0.5).float().cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(pet_np,  cmap="hot");  axes[0].set_title("PET input")
axes[1].imshow(gt_mask, cmap="gray"); axes[1].set_title("GT Mask")
axes[2].imshow(dl_pred, cmap="gray"); axes[2].set_title(f"DL  (Dice={dl_dices[i]:.2f})")
axes[3].imshow(thr_mask,cmap="gray"); axes[3].set_title(f"40% Threshold (Dice={thr_dices[i]:.2f})")
for ax in axes:
    ax.axis("off")
plt.suptitle("DL Model vs 40 % SUV Threshold Baseline", fontsize=13)
plt.tight_layout()
plt.show()